# Module 0 - Setup

In this notebook you install the Python packages for the workshop and turn on model invocation logging.

## Contents

1. [Introduction](#intro)
2. [Install the Python packages](#install)
3. [Turn on model invocation logging](#logging)
4. [Confirm that logging works](#confirm)
5. [Try it](#try-it)
6. [Conclusion and key takeaways](#conclusion)
7. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

Over this workshop you build a travel assistant for AnyCompany, a fictional company, on Amazon Bedrock. Before you start, this notebook does two things:

| Step | Why |
|:--|:--|
| Install the Python packages | The notebooks use the AWS SDK for Python (boto3), the Anthropic SDK and the OpenAI SDK |
| Turn on model invocation logging | Records every model call you make, so you can review them in Module 6 |

You turn logging on now, at the start, so that by Module 6 you have a record of every call from the whole workshop to analyze.

## 2. Install the Python packages <a id="install"></a>

The packages are listed in `requirements.txt` at the root of the repository. Run the cell below to install them. It takes up to a minute.

In [ ]:
%pip install --quiet -r ../requirements.txt

## 3. Turn on model invocation logging <a id="logging"></a>

[Model invocation logging](https://docs.aws.amazon.com/bedrock/latest/userguide/model-invocation-logging.html) records every model call in your account and Region: who made it, which model, the input and output, token counts, and any metadata you attach. It is off by default. In Module 6 you use these logs to monitor the deployed assistant and attribute cost.

Logging needs three things, which you set up in the next three cells:

1. A **CloudWatch Logs log group** to store the records.
2. An **IAM role** that lets Amazon Bedrock write to that log group.
3. The **logging configuration**, which turns logging on and points it at the log group.

> **Important:** The logs contain the full text of prompts and responses. In production, decide who can read the log group and how long to keep the records. This workshop keeps them for 7 days.

Run the cell below to create the clients and the log group.

In [ ]:
import json
import time

import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

logs = boto3.client("logs", region_name=REGION)
iam = boto3.client("iam")
bedrock = boto3.client("bedrock", region_name=REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

LOG_GROUP_NAME = "/aim317/bedrock-invocations"
LOGGING_ROLE_NAME = "aim317-bedrock-logging-role"

try:
    logs.create_log_group(logGroupName=LOG_GROUP_NAME)
    print(f"Created log group {LOG_GROUP_NAME}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"Log group {LOG_GROUP_NAME} already exists")

logs.put_retention_policy(logGroupName=LOG_GROUP_NAME, retentionInDays=7)

Next, the IAM role. Its **trust policy** says who can use the role: Amazon Bedrock, acting for your account only. Its **permissions** say what the role can do: write log records to the workshop log group, and nothing else.

Run the cell below to create the role.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {
            "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
            "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:*"},
        },
    }],
}
permissions = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["logs:CreateLogStream", "logs:PutLogEvents"],
        "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LOG_GROUP_NAME}:log-stream:*",
    }],
}

try:
    LOGGING_ROLE_ARN = iam.create_role(
        RoleName=LOGGING_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="AIM317 workshop: lets Amazon Bedrock write invocation logs",
    )["Role"]["Arn"]
    print(f"Created role {LOGGING_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    LOGGING_ROLE_ARN = iam.get_role(RoleName=LOGGING_ROLE_NAME)["Role"]["Arn"]
    print(f"Role {LOGGING_ROLE_NAME} already exists")

iam.put_role_policy(
    RoleName=LOGGING_ROLE_NAME,
    PolicyName="write-invocation-logs",
    PolicyDocument=json.dumps(permissions),
)
print(LOGGING_ROLE_ARN)

Finally, turn logging on. The configuration applies to every model call in this account and Region. `textDataDeliveryEnabled` records the text of prompts and responses; this workshop does not use images, video or embeddings, so those stay off.

A new IAM role can take a few seconds to become usable, so the cell retries until Amazon Bedrock accepts it.

In [ ]:
for attempt in range(10):
    try:
        bedrock.put_model_invocation_logging_configuration(
            loggingConfig={
                "cloudWatchConfig": {"logGroupName": LOG_GROUP_NAME, "roleArn": LOGGING_ROLE_ARN},
                "textDataDeliveryEnabled": True,
                "imageDataDeliveryEnabled": False,
                "embeddingDataDeliveryEnabled": False,
                "videoDataDeliveryEnabled": False,
            }
        )
        break
    except bedrock.exceptions.ValidationException as error:
        print(f"Waiting for the role to be ready ({error.response['Error']['Message'][:80]}...)")
        time.sleep(10)

config = bedrock.get_model_invocation_logging_configuration()["loggingConfig"]
print("Logging to:", config["cloudWatchConfig"]["logGroupName"])
print("Text data: ", config["textDataDeliveryEnabled"])

## 4. Confirm that logging works <a id="confirm"></a>

Run the cell below to make a model call and wait for its log record, which usually arrives within a minute. The call includes `requestMetadata`: tags you attach to a request, which appear in its log record. In Module 6 you use tags like these to see which part of your application made each call.

In [ ]:
response = bedrock_runtime.converse(
    modelId="openai.gpt-oss-120b-1:0",
    messages=[{"role": "user", "content": [{"text": "Say hello to the AnyCompany travel team."}]}],
    inferenceConfig={"maxTokens": 500},
    requestMetadata={"workshop": "aim317", "module": "00-setup"},
)
request_id = response["ResponseMetadata"]["RequestId"]
print("Request ID:", request_id)

record = None
started = time.time()
while record is None and time.time() - started < 180:
    time.sleep(10)
    events = logs.filter_log_events(
        logGroupName=LOG_GROUP_NAME,
        startTime=int((started - 60) * 1000),
        filterPattern=f'{{ $.requestId = "{request_id}" }}',
    )["events"]
    if events:
        record = json.loads(events[0]["message"])

if record:
    print(f"Log record arrived after {time.time() - started:.0f} seconds:\n")
    print("Operation:       ", record["operation"])
    print("Model:           ", record["modelId"])
    print("Caller:          ", record["identity"]["arn"])
    print("Request metadata:", record["requestMetadata"])
    print("Input tokens:    ", record["input"]["inputTokenCount"])
    print("Output tokens:   ", record["output"]["outputTokenCount"])
else:
    print("No log record yet. Records can take a few minutes; you check the logs again in Module 6.")

Every call you make from now on is recorded the same way.

## 5. Try it <a id="try-it"></a>

1. Open the [CloudWatch Logs console](https://console.aws.amazon.com/cloudwatch/home#logsV2:log-groups), choose the `/aim317/bedrock-invocations` log group, and find the record for your call. Expand `input` and `output` to see the full prompt and response.
2. Run the cell below to print the whole record. Which fields would help you investigate a slow or expensive request?

In [ ]:
print(json.dumps(record, indent=2) if record else "Run the cell in section 4 first.")

## 6. Conclusion and key takeaways <a id="conclusion"></a>

- The workshop's Python packages are listed in `requirements.txt`.
- **Model invocation logging** records every call, including prompts, responses, tokens, the caller and request metadata. It needs a log group, an IAM role and one configuration call.
- Turning logging on at the start means you have a complete record to monitor and analyze later.

## 7. Next steps <a id="next-steps"></a>

Your environment is ready. In [Module 1 - First Call](01_first_call.ipynb), you make your first Amazon Bedrock call with the Converse API and build the first version of AnyCompany's travel assistant.